# 01 — Deep Agents 沙箱

**来源:** [LangChain Docs — Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)

沙箱为 Agent 提供隔离的执行环境，防止 Agent 访问宿主机上的凭据、文件或网络。

在 Deep Agents 中，沙箱是一种特殊的 **Backend**。与其他 Backend（State、Filesystem、Store）只暴露文件操作不同，沙箱 Backend 额外提供 `execute` 工具，让 Agent 可以在隔离环境中运行 Shell 命令。

## 1. 架构概览

Agent 通过 Backend 协议与沙箱通信，沙箱内部提供文件系统、Bash Shell 和依赖管理：

```mermaid
graph LR
    subgraph Agent
        LLM --> Tools
        Tools --> LLM
    end
    Agent <-- backend protocol --> Sandbox
    subgraph Sandbox
        Filesystem
        Bash
        Dependencies
    end
```

**沙箱提供的工具：**
- 标准文件系统工具：`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- `execute` — 在沙箱中运行任意 Shell 命令
- 安全的隔离边界，保护宿主机

## 2. 为什么使用沙箱？

| 场景 | 说明 |
|------|------|
| **编码 Agent** | 使用 Shell、Git、克隆仓库、Docker-in-Docker 构建测试 |
| **数据分析 Agent** | 加载文件、安装库（pandas/numpy）、运行统计、生成报告 |
| **Deep Agents Code** | 内置 `--sandbox` 标志，见 [Use remote sandboxes](https://docs.langchain.com/oss/python/deepagents/code/remote-sandboxes) |

沙箱不仅是安全边界，也是 **Agent 操作环境的统一抽象层**。

## 3. 可用沙箱 Backend

| Backend | 包 | 特点 |
|---------|-----|------|
| **Modal** | `langchain-modal` | 按需 Serverless 沙箱，用完即销毁 |
| **Runloop** | `langchain-runloop` | Devbox 模式，持久化开发环境 |
| **Daytona** | `langchain-daytona` | 轻量、快速创建/销毁 |
| **LangSmith** | `langchain-langsmith` | LangSmith 平台集成 |
| **AgentCore** | `langchain-agentcore` | AgentCore 后端 |
| **LocalShell** | 内置 | 本地 Shell 执行（无隔离） |

### 3.1 Modal

In [ ]:
# 安装: pip install langchain-modal

import modal
from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain_modal import ModalSandbox
from dotenv import load_dotenv
load_dotenv()

app = modal.App.lookup("your-app")
modal_sandbox = modal.Sandbox.create(app=app)
backend = ModalSandbox(sandbox=modal_sandbox)

agent = create_deep_agent(
    model=ChatAnthropic(model="deepseek-v4-flash"),
    system_prompt="你是一名 Python 编程助手 with sandbox access.",
    backend=backend,
)

try:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "创建一个 Python 包并运行测试"}]
    })
finally:
    modal_sandbox.terminate()  # 用完一定要销毁


### 3.2 Runloop

In [ ]:
# 安装: pip install langchain-runloop

import os
from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain_runloop import RunloopSandbox
from runloop_api_client import RunloopSDK

client = RunloopSDK(bearer_token=os.environ["RUNLOOP_API_KEY"])
devbox = client.devbox.create()
backend = RunloopSandbox(devbox=devbox)

agent = create_deep_agent(
    model=ChatAnthropic(model="deepseek-v4-flash"),
    system_prompt="你是一名 Python 编程助手 with sandbox access.",
    backend=backend,
)

try:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "创建一个 Python 包并运行测试"}]
    })
finally:
    devbox.shutdown()

### 3.3 Daytona

In [ ]:
# 安装: pip install langchain-daytona

from daytona import Daytona
from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

agent = create_deep_agent(
    model=ChatAnthropic(model="deepseek-v4-flash"),
    system_prompt="你是一名 Python 编程助手 with sandbox access.",
    backend=backend,
)

try:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "创建一个 Python 包并运行测试"}]
    })
finally:
    sandbox.stop()

### 3.4 LangSmith Sandbox

In [ ]:
# 安装: pip install langchain-langsmith

from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain_langsmith import LangSmithSandbox

backend = LangSmithSandbox()

agent = create_deep_agent(
    model=ChatAnthropic(model="deepseek-v4-flash"),
    system_prompt="你是一名 Python 编程助手 with sandbox access.",
    backend=backend,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "创建一个 Python 包并运行测试"}]
})

### 3.5 AgentCore

In [ ]:
# 安装: pip install langchain-agentcore

from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain_agentcore import AgentCoreSandbox

backend = AgentCoreSandbox()

agent = create_deep_agent(
    model=ChatAnthropic(model="deepseek-v4-flash"),
    system_prompt="你是一名 Python 编程助手 with sandbox access.",
    backend=backend,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "创建一个 Python 包并运行测试"}]
})

### 3.6 LocalShellBackend

直接在宿主机上执行 Shell 命令（**无隔离**，仅用于本地开发调试）：

In [ ]:
from deepagents.backends import LocalShellBackend

backend = LocalShellBackend()
# backend = LocalShellBackend(workdir="/workspace")  # 指定工作目录

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
)

## 4. 沙箱 Backend 接口

沙箱 Backend 需要实现 `SandboxBackendProtocolV2`：

| 方法 | 说明 |
|------|------|
| `exec_command(cmd, timeout, workdir)` | 执行 Shell 命令 |
| `read_file(path, offset, limit)` | 读取文件 |
| `write_file(path, content)` | 写入文件 |
| `edit_file(path, old, new)` | 编辑文件 |
| `list(path)` | 列出目录 |
| `glob(pattern)` | 查找文件 |
| `grep(pattern, path)` | 搜索内容 |
| `file_stat(path)` | 文件状态 |
| `connect()` / `disconnect()` | 连接/断开沙箱 |
| `get_sandbox_name()` | 沙箱名称 |

当 Harness 检测到 Backend 实现了 `SandboxBackendProtocolV2`，会自动添加 `execute` 工具。

## 5. 沙箱生命周期

典型的沙箱使用模式：

```python
# 1. 创建沙箱实例（API 调用）
sandbox = ProviderSandboxClass.create()

# 2. 包装为 Deep Agents Backend
backend = SandboxBackendAdapter(sandbox=sandbox)

# 3. 关联 Agent
agent = create_deep_agent(..., backend=backend)

# 4. 使用 Agent（自动通过 Backend 执行所有文件/命令操作）
result = agent.invoke(...)

# 5. 清理
sandbox.terminate()  # 或 sandbox.stop() / devbox.shutdown()
```

> **重要：** 务必在 `try/finally` 中确保沙箱被销毁，否则会产生遗留费用。

## 6. 文件传输与结果获取

部分沙箱 Provider 提供独立的文件传输 API：

| Provider | 上传 | 下载 |
|----------|------|------|
| Daytona | `sandbox.files.upload(path, content)` | `sandbox.files.download(path)` |
| Runloop | `devbox.upload(path, content)` | `devbox.download(path)` |
| Modal | `sandbox.put_file(path, content)` | `sandbox.get_file(path)` |
| LangSmith | `backend.transfer(agent_id, op, path, content)` | 同上 |

---
**参考链接**

- [Sandboxes — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/sandboxes)
- [Backends](https://docs.langchain.com/oss/python/deepagents/backends)
- [Deep Agents Code — Remote Sandboxes](https://docs.langchain.com/oss/python/deepagents/code/remote-sandboxes)